# Retrieval Augmented Generation Demo
## Asian Women Advancing AI (AWAAI)
### Zeba Karkhanawala

### What are we going to build? A Quick Overview

- We take a document (like an open book, here a PDF) and extract its text.  
- Just like dividing a book into paragraphs or sections for easier reference, we split the document into smaller, manageable text chunks.  
- We convert these chunks into numerical representations (embeddings) and store them in a **vector database** (ChromaDB) for efficient retrieval.  
- When you ask a question, instead of scanning the whole book, we retrieve the **most relevant sections** based on similarity to your query.  
- The retrieved sections are provided as **context** augmented with your question to ensure the answer is grounded in real information.  
- A **GPT-4 model** processes the query and context together, generating a response **just like an open-book exam answer** where only relevant information is used.  
- This approach ensures that the answers are **relevant, fact-based, and context-aware**, making it more reliable than a model generating responses from memory alone. 🚀

### Installing and Loading the Libraries

This code block installs several Python packages using pip. Here's what each package does:

openai:
- Provides access to OpenAI's APIs, including GPT models.
- Used for generating text, embeddings, and other AI-related tasks.

langchain:
- A framework for building applications using Large Language Models (LLMs).
- Supports tasks like retrieval-augmented generation, chaining prompts, and managing context.

langchain-community:
- Community-contributed extensions or modules for the langchain library.
- Adds experimental or community-specific features to enhance functionality.

pypdf:
- A library for reading and manipulating PDF files.
- Useful for extracting text or metadata from PDFs.

tiktoken:
- A tokenizer library for OpenAI models.
Helps in managing token counts, ensuring inputs fit within the model's token limit.

chromadb:
- A vector database for managing embeddings and similarity search.
- Commonly used in retrieval-augmented generation workflows to store and retrieve vectorized data.


In [10]:
%pip install openai langchain langchain-community pypdf tiktoken chromadb

This code block imports the necessary libraries and initializes the OpenAI API key. Make sure to replace the 'your-api-key' with your own OpenAI API key before running!

In [16]:
# Import necessary libraries
import openai
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA

# Initialize OpenAI API (Make sure to replace 'your-api-key' with your actual API key)
openai.api_key = 'your-api-key'

### Loading Dataset

This code uses the **`PyPDFLoader`** class from `langchain.document_loaders` to load and process a PDF file named **`MyLinkedInProfile.pdf`**. The `pdf_loader.load()` method extracts the text content from the PDF and stores it in the variable **`documents`** as a list of text chunks. This is a crucial step for preparing the PDF data for further processing, such as splitting text, generating embeddings, or performing question-answering tasks. Ensure the file path is correct and the file is accessible to avoid errors.

In [17]:
# Load and process the PDF
pdf_loader = PyPDFLoader("your-pdf-file-path.pdf")
documents = pdf_loader.load()

### Chunking the Data
This code initializes a **`RecursiveCharacterTextSplitter`** to divide text data into smaller, manageable chunks. The splitter is configured with a **`chunk_size`** of 500 characters and an **`chunk_overlap`** of 100 characters, ensuring that overlapping content is retained between chunks to preserve context. The **`split_documents`** method is then applied to the **`documents`** variable (loaded from the PDF), resulting in a list of **`text_chunks`**. The total number of chunks created is printed using **`len(text_chunks)`**, providing insight into how the text has been segmented for downstream tasks like embedding generation or question-answering.

In [18]:
# Initialize a text splitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)

# Split the text into chunks
text_chunks = text_splitter.split_documents(documents)

print(f"Number of chunks from the text: {len(text_chunks)}")

Number of chunks from the text: 26


### Creating Vector Embeddings & storing them in a Vector Database

This code initializes an **OpenAI embeddings model** using the **`OpenAIEmbeddings`** class and the API key previously set in `openai.api_key`. The embeddings model is then used to generate vector representations for the document chunks.

Next, it stores these vectors in a **Chroma vector store** by calling **`Chroma.from_documents`**, passing the `text_chunks` and the `embedding_model`. This allows for efficient similarity searches later on.

Finally, the **retriever** is initialized using **`vector_store.as_retriever`**, which configures the Chroma store to retrieve the top 3 most relevant document chunks based on similarity (specified by `search_type="similarity"` and `search_kwargs={"k": 3}`). This setup is essential for performing efficient document retrieval in response to queries, leveraging the stored embeddings for quick, context-aware results.

In [20]:
# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings(api_key=openai.api_key)

# Storing the documents in a Chroma vector store
vector_store = Chroma.from_documents(documents=text_chunks, embedding=embedding_model)

# Initializing the retriever
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 3})

### Retrieving Relevant Documents

This code defines a **query**: `"What are Zeba's Top Skills?"`, which will be used to search for relevant information in the previously stored document chunks. The **retriever** (initialized in the previous code block) is then used to perform a similarity search with **`retriever.get_relevant_documents(query)`**. This method returns a list of **`relevant_docs`**, which are the top documents most similar to the query based on the embeddings stored in the Chroma vector store. The retrieved documents are likely to contain information related to Zeba’s skills, enabling further processing like generating an answer to the query.

In [21]:
# question
query = input("Enter your question: ")

# Retrieve documents relevant to the question
relevant_docs = retriever.get_relevant_documents(query)

### Augmenting the Retrieved Documents to the Prompt

This code augments the retrieved documents by combining them with the original query, creating a **contextual query**. The **`contextual_query`** variable concatenates the relevant documents (stored in **`relevant_docs`**) with the query `"What are Zeba's Top Skills?"` in a readable format: the context (the documents) is followed by the query.

The **`prompt_with_context`** is then created as a list of dictionaries that structure the conversation for the large language model (LLM). The system message instructs the model to act as an expert assistant that answers the query based on the provided context. The user message includes the **`contextual_query`**, which gives the LLM both the retrieved information and the question it needs to answer. This setup ensures the LLM can generate a response that is grounded in the relevant context retrieved earlier.

In [22]:
# Augmenting retrieved document with the query - giving the retrieved document as the context with the question we have
contextual_query = f"Context: {relevant_docs}\n\nQuery: {query}"

# creating the prompts to pass to the LLM
# this prompt mentions the context which is the retrieved document
prompt_with_context = [
    {"role": "system", "content": "You are an expert assistant that answers the query based on the context provided."},
    {"role": "user", "content": contextual_query}
]

### Generating a final response using an LLM

This code generates an answer to the query using OpenAI's GPT model. The **`openai.chat.completions.create`** method is called with the **`model="gpt-4o"`** parameter, indicating the use of OpenAI's GPT-4 model (with the `gpt-4o` variant). The **`messages`** argument is passed the **`prompt_with_context`**, which contains both the query and the relevant context.

The response is stored in **`text_response`**, and the answer generated by the model is extracted from the response using **`text_response.choices[0].message.content`**. This provides the content of the model's reply.

Finally, the code prints both the original query and the model's answer (**`answer_with_context`**), ensuring that the response is clearly displayed for the user. The generated answer is expected to be informed by the context provided in the prompt.

In [23]:
# Generate answer using OpenAI GPT model
text_response = openai.chat.completions.create(
  model="gpt-4o",
  messages=prompt_with_context,
)

# cleaning up the response given by the LLM
answer_with_context = text_response.choices[0].message.content

# printing the answer
print(f"Query: {query}")
print(f"Answer: {answer_with_context}")

Query: What are Zeba's Top Skills?
Answer: Zeba's top skills are PySpark, AI Agents, and Neural Networks.
